## Multi-Datasets experiment


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

In [3]:
def sample_and_evaluate(X, y, K, test_size=200, seed=None):
    """
    Sample K examples per class for training, test_size for testing.
    Train logistic regression (no regularization), return accuracy and effective rank.
    """
    rng = np.random.default_rng(seed)
    
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]
    
    # Safety check
    if K > len(idx_0) - test_size or K > len(idx_1) - test_size:
        raise ValueError(f"K={K} too large for available data")
    
    train_0 = rng.choice(idx_0, size=K, replace=False)
    train_1 = rng.choice(idx_1, size=K, replace=False)
    
    rem_0 = np.setdiff1d(idx_0, train_0)
    rem_1 = np.setdiff1d(idx_1, train_1)
    
    test_0 = rng.choice(rem_0, size=test_size, replace=False)
    test_1 = rng.choice(rem_1, size=test_size, replace=False)
    
    X_train = np.vstack([X[train_0], X[train_1]])
    y_train = np.array([0]*K + [1]*K)
    X_test = np.vstack([X[test_0], X[test_1]])
    y_test = np.array([0]*test_size + [1]*test_size)
    
    # Center training data
    X_train_mean = X_train.mean(axis=0)
    X_train_centered = X_train - X_train_mean
    X_test_centered = X_test - X_train_mean
    
    # Train (C=np.inf = no regularization)
    clf = LogisticRegression(max_iter=5000, C=np.inf)
    clf.fit(X_train_centered, y_train)
    acc = clf.score(X_test_centered, y_test)
    
    # Pooled within-class covariance
    cov_0 = np.cov(X_train_centered[:K], rowvar=False, bias=True)
    cov_1 = np.cov(X_train_centered[K:], rowvar=False, bias=True)
    cov_pooled = 0.5 * (cov_0 + cov_1)
    
    # Effective rank
    eigvals = np.linalg.eigvalsh(cov_pooled)
    eigvals = np.maximum(eigvals, 1e-12)
    p = eigvals / eigvals.sum()
    erank = np.exp(-np.sum(p * np.log(p)))
    
    return acc, erank

In [4]:
def run_experiment(name, X, y, class_a, class_b, Ks, test_size, n_trials=50, pca_dims=50):
    """
    Run full K-sweep for one dataset and one class pair.
    Returns list of dicts with K, mean_acc, std_acc, mean_erank, S.
    """
    # Filter and relabel
    mask = (y == class_a) | (y == class_b)
    X_pair = X[mask].astype(np.float64)
    y_pair = y[mask]
    y_pair = np.where(y_pair == class_a, 0, 1)
    
    # Preprocessing
    if pca_dims is not None:
        scaler = StandardScaler()
        X_pair = scaler.fit_transform(X_pair)
        pca = PCA(n_components=pca_dims)
        X_pair = pca.fit_transform(X_pair)
    else:
        scaler = StandardScaler()
        X_pair = scaler.fit_transform(X_pair)
    
    results = []
    print(f"\n{'='*60}")
    print(f"{name} | {class_a} vs {class_b} | {n_trials} trials")
    print(f"{'='*60}")
    
    for K in Ks:
        n0 = (y_pair == 0).sum()
        n1 = (y_pair == 1).sum()
        if K > min(n0, n1) - test_size:
            print(f"  K={K:4d}: SKIPPED (insufficient data: {n0}/{n1} available)")
            continue
            
        accs, eranks = [], []
        for trial in range(n_trials):
            acc, erank = sample_and_evaluate(X_pair, y_pair, K=K, test_size=test_size, seed=trial)
            accs.append(acc)
            eranks.append(erank)
        
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        mean_erank = np.mean(eranks)
        S = mean_erank / K
        
        results.append({
            'K': K, 'mean_acc': mean_acc, 'std_acc': std_acc,
            'mean_erank': mean_erank, 'S': S,
            'marginal': mean_acc - results[-1]['mean_acc'] if results else 0
        })
        print(f"  K={K:4d}: acc={mean_acc:.4f}±{std_acc:.4f}, "
              f"erank={mean_erank:.2f}, S={S:.4f}, marginal={results[-1]['marginal']:+.4f}")
    
    return results

In [5]:
# Load from pre-downloaded npz files (instant)
print("Loading MNIST...")
mnist_data = np.load('mnist.npz')
X_mnist = mnist_data['X'] / 255.0
y_mnist = mnist_data['y']

print("Loading Fashion-MNIST...")
fashion_data = np.load('fashion_mnist.npz')
X_fashion = fashion_data['X'] / 255.0
y_fashion = fashion_data['y']

print("Loading Kuzushiji-MNIST...")
kuzu_data = np.load('kuzushiji.npz')
X_kuzushiji = kuzu_data['X'] / 255.0
y_kuzushiji = kuzu_data['y']

print("Loading USPS...")
usps_data = np.load('usps.npz')
X_usps = usps_data['X']
y_usps = usps_data['y']

print("Loading Breast Cancer...")
bc_data = np.load('breast_cancer.npz')
X_bc = bc_data['X']
y_bc = bc_data['y']

print("All datasets loaded.")

Loading MNIST...
Loading Fashion-MNIST...
Loading Kuzushiji-MNIST...
Loading USPS...
Loading Breast Cancer...
All datasets loaded.


### MNIST

In [6]:
# MNIST has ~6000 per class. We can go to K=4096 with test_size=200.
# We include THREE pairs to show the decoupling: difficulty ≠ geometric complexity

mnist_pairs = [
    ('MNIST_0v1', 0, 1, 'easy'),      # visually very distinct
    ('MNIST_3v8', 3, 8, 'medium'),      # your main pair
    ('MNIST_4v9', 4, 9, 'hard'),        # visually similar, but we found LOWER erank!
]

mnist_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

results_mnist = {}
for name, a, b, difficulty in mnist_pairs:
    results_mnist[name] = run_experiment(
        name, X_mnist, y_mnist, a, b,
        Ks=mnist_Ks, test_size=200, n_trials=50, pca_dims=50
    )


MNIST_0v1 | 0 vs 1 | 50 trials
  K=   2: acc=0.8896±0.1050, erank=1.65, S=0.8239, marginal=+0.0000
  K=   4: acc=0.9594±0.0491, erank=3.76, S=0.9407, marginal=+0.0697
  K=   8: acc=0.9860±0.0151, erank=6.67, S=0.8332, marginal=+0.0266
  K=  16: acc=0.9909±0.0068, erank=9.85, S=0.6156, marginal=+0.0049
  K=  32: acc=0.9936±0.0053, erank=12.98, S=0.4056, marginal=+0.0027
  K=  64: acc=0.9952±0.0043, erank=15.66, S=0.2447, marginal=+0.0016
  K= 128: acc=0.9947±0.0050, erank=17.38, S=0.1358, marginal=-0.0006
  K= 256: acc=0.9955±0.0035, erank=20.19, S=0.0789, marginal=+0.0009
  K= 512: acc=0.9951±0.0038, erank=23.06, S=0.0450, marginal=-0.0004
  K=1024: acc=0.9966±0.0030, erank=26.65, S=0.0260, marginal=+0.0014
  K=2048: acc=0.9969±0.0028, erank=29.60, S=0.0145, marginal=+0.0003
  K=4096: acc=0.9966±0.0035, erank=32.24, S=0.0079, marginal=-0.0003

MNIST_3v8 | 3 vs 8 | 50 trials
  K=   2: acc=0.6988±0.1000, erank=1.80, S=0.8984, marginal=+0.0000
  K=   4: acc=0.7967±0.0571, erank=4.44, S=1

### Fashion-MNIST

In [7]:
# Fashion-MNIST: same 784D structure, different visual domain
# 0=T-shirt, 1=Trouser (easy), 2=Pullover, 6=Shirt (hard)

fashion_pairs = [
    ('Fashion_0v1', 0, 1, 'easy'),
    ('Fashion_2v6', 2, 6, 'hard'),
]

fashion_Ks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

results_fashion = {}
for name, a, b, difficulty in fashion_pairs:
    results_fashion[name] = run_experiment(
        name, X_fashion, y_fashion, a, b,
        Ks=fashion_Ks, test_size=200, n_trials=50, pca_dims=50
    )


Fashion_0v1 | 0 vs 1 | 50 trials
  K=   2: acc=0.8520±0.0800, erank=1.68, S=0.8376, marginal=+0.0000
  K=   4: acc=0.9078±0.0628, erank=3.56, S=0.8910, marginal=+0.0559
  K=   8: acc=0.9304±0.0342, erank=5.55, S=0.6932, marginal=+0.0226
  K=  16: acc=0.9455±0.0170, erank=7.84, S=0.4902, marginal=+0.0151
  K=  32: acc=0.9608±0.0163, erank=9.57, S=0.2990, marginal=+0.0153
  K=  64: acc=0.9639±0.0123, erank=12.00, S=0.1875, marginal=+0.0031
  K= 128: acc=0.9727±0.0103, erank=13.76, S=0.1075, marginal=+0.0087
  K= 256: acc=0.9748±0.0101, erank=16.10, S=0.0629, marginal=+0.0021
  K= 512: acc=0.9724±0.0098, erank=17.91, S=0.0350, marginal=-0.0023
  K=1024: acc=0.9781±0.0076, erank=19.14, S=0.0187, marginal=+0.0057
  K=2048: acc=0.9839±0.0068, erank=20.25, S=0.0099, marginal=+0.0058
  K=4096: acc=0.9851±0.0057, erank=20.69, S=0.0051, marginal=+0.0011

Fashion_2v6 | 2 vs 6 | 50 trials
  K=   2: acc=0.5547±0.0633, erank=1.71, S=0.8564, marginal=+0.0000
  K=   4: acc=0.6160±0.0842, erank=3.50, 

### Kuzushiji-MNIST

In [8]:
# Japanese characters. Completely different alphabet.
results_kuzushiji = {
    'Kuzushiji_0v1': run_experiment(
        'Kuzushiji_0v1', X_kuzushiji, y_kuzushiji, 0, 1,
        Ks=[2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096],
        test_size=200, n_trials=50, pca_dims=50
    )
}


Kuzushiji_0v1 | 0 vs 1 | 50 trials
  K=   2: acc=0.8667±0.1073, erank=1.82, S=0.9100, marginal=+0.0000
  K=   4: acc=0.9250±0.0337, erank=4.76, S=1.1905, marginal=+0.0583
  K=   8: acc=0.9409±0.0205, erank=9.66, S=1.2070, marginal=+0.0159
  K=  16: acc=0.9529±0.0200, erank=16.47, S=1.0292, marginal=+0.0121
  K=  32: acc=0.9631±0.0167, erank=23.60, S=0.7375, marginal=+0.0102
  K=  64: acc=0.9661±0.0138, erank=29.56, S=0.4619, marginal=+0.0030
  K= 128: acc=0.9678±0.0117, erank=33.01, S=0.2579, marginal=+0.0017
  K= 256: acc=0.9699±0.0103, erank=35.12, S=0.1372, marginal=+0.0021
  K= 512: acc=0.9719±0.0091, erank=36.08, S=0.0705, marginal=+0.0020
  K=1024: acc=0.9817±0.0054, erank=36.80, S=0.0359, marginal=+0.0098
  K=2048: acc=0.9860±0.0060, erank=37.06, S=0.0181, marginal=+0.0043
  K=4096: acc=0.9873±0.0057, erank=37.19, S=0.0091, marginal=+0.0014


### Breast Cancer


In [9]:
# Breast Cancer: 30D tabular medical data. NOT images. NO PCA.
# Imbalanced: 212 malignant (0), 357 benign (1)
# Max K = 100 (limited by minority class 212 - 50 test = 162)

results_bc = {
    'BreastCancer': run_experiment(
        'BreastCancer', X_bc, y_bc, 0, 1,
        Ks=[2, 4, 8, 16, 32, 64, 100],
        test_size=50, n_trials=100, pca_dims=None
    )
}


BreastCancer | 0 vs 1 | 100 trials
  K=   2: acc=0.8381±0.0784, erank=1.69, S=0.8427, marginal=+0.0000
  K=   4: acc=0.8807±0.0629, erank=3.41, S=0.8535, marginal=+0.0426
  K=   8: acc=0.9097±0.0455, erank=5.03, S=0.6287, marginal=+0.0290
  K=  16: acc=0.9197±0.0386, erank=6.44, S=0.4024, marginal=+0.0100
  K=  32: acc=0.9370±0.0306, erank=7.21, S=0.2252, marginal=+0.0173
  K=  64: acc=0.9449±0.0225, erank=7.91, S=0.1236, marginal=+0.0079
  K= 100: acc=0.9425±0.0212, erank=8.20, S=0.0820, marginal=-0.0024


### USPS

In [10]:
usps = fetch_openml('usps', version=1, parser='auto', as_frame=False)
X_usps = usps.data.astype(np.float64)
y_usps = usps.target.astype(int)  # <-- THIS WAS MISSING

print(f"USPS loaded: {X_usps.shape}, labels: {np.unique(y_usps)}")

/opt/anaconda3/envs/spectral-saturation/lib/python3.13/site-packages/sklearn/datasets/_openml.py:1035: UserWarning: Version 1 of dataset USPS is inactive, meaning that issues have been found in the dataset. Try using a newer version from this URL: https://openml.org/data/v1/download/18805612/USPS.arff
  warn(


USPS loaded: (9298, 256), labels: [ 1  2  3  4  5  6  7  8  9 10]


In [11]:
# USPS: labels are 1-10, not 0-9
results_usps = {
    'USPS_1v2': run_experiment(
        'USPS_1v2', X_usps, y_usps, 1, 2,  # <-- use 1 and 2, not 0 and 1
        Ks=[2, 4, 8, 16, 32, 64, 128, 256, 512],
        test_size=100, n_trials=50, pca_dims=50
    )
}


USPS_1v2 | 1 vs 2 | 50 trials
  K=   2: acc=0.9366±0.0713, erank=1.39, S=0.6960, marginal=+0.0000
  K=   4: acc=0.9733±0.0313, erank=3.06, S=0.7656, marginal=+0.0367
  K=   8: acc=0.9860±0.0226, erank=5.27, S=0.6586, marginal=+0.0127
  K=  16: acc=0.9945±0.0067, erank=7.61, S=0.4756, marginal=+0.0085
  K=  32: acc=0.9970±0.0037, erank=9.99, S=0.3121, marginal=+0.0025
  K=  64: acc=0.9969±0.0041, erank=11.74, S=0.1835, marginal=-0.0001
  K= 128: acc=0.9972±0.0036, erank=13.48, S=0.1053, marginal=+0.0003
  K= 256: acc=0.9973±0.0036, erank=14.24, S=0.0556, marginal=+0.0001
  K= 512: acc=0.9973±0.0038, erank=15.08, S=0.0295, marginal=+0.0000


In [12]:
# Combine everything
all_results = {}
all_results.update(results_mnist)
all_results.update(results_fashion)
all_results.update(results_kuzushiji)
all_results.update(results_usps)
all_results.update(results_bc)

print(f"\nTotal experiments: {len(all_results)}")
for name, res in all_results.items():
    print(f"  {name}: {len(res)} K values")


Total experiments: 8
  MNIST_0v1: 12 K values
  MNIST_3v8: 12 K values
  MNIST_4v9: 12 K values
  Fashion_0v1: 12 K values
  Fashion_2v6: 12 K values
  Kuzushiji_0v1: 12 K values
  USPS_1v2: 9 K values
  BreastCancer: 7 K values


### CIFAR-10

In [ ]:
from tensorflow.keras.datasets import cifar10

(X_cifar, y_cifar), (_, _) = cifar10.load_data()
X_cifar = X_cifar.reshape(-1, 3072).astype(np.float64) / 255.0
y_cifar = y_cifar.flatten()



In [17]:
# CIFAR-10 extended to K=4096
results_cifar = {
    'CIFAR_0v1': run_experiment('CIFAR_0v1', X_cifar, y_cifar, 0, 1,
        Ks=[2,4,8,16,32,64,128,256,512,1024,2048,4096], test_size=200, n_trials=50, pca_dims=50),
    'CIFAR_3v5': run_experiment('CIFAR_3v5', X_cifar, y_cifar, 3, 5,
        Ks=[2,4,8,16,32,64,128,256,512,1024,2048,4096], test_size=200, n_trials=50, pca_dims=50)
}



CIFAR_0v1 | 0 vs 1 | 50 trials
  K=   2: acc=0.6142±0.0903, erank=1.82, S=0.9095, marginal=+0.0000
  K=   4: acc=0.6218±0.0600, erank=4.24, S=1.0597, marginal=+0.0076
  K=   8: acc=0.6500±0.0548, erank=7.31, S=0.9139, marginal=+0.0282
  K=  16: acc=0.6841±0.0454, erank=10.00, S=0.6251, marginal=+0.0341
  K=  32: acc=0.7058±0.0382, erank=12.43, S=0.3884, marginal=+0.0217
  K=  64: acc=0.6932±0.0368, erank=13.75, S=0.2149, marginal=-0.0126
  K= 128: acc=0.7436±0.0263, erank=14.53, S=0.1135, marginal=+0.0504
  K= 256: acc=0.7787±0.0205, erank=14.86, S=0.0581, marginal=+0.0352
  K= 512: acc=0.7928±0.0176, erank=15.35, S=0.0300, marginal=+0.0141
  K=1024: acc=0.7999±0.0164, erank=15.38, S=0.0150, marginal=+0.0071
  K=2048: acc=0.8008±0.0144, erank=15.35, S=0.0075, marginal=+0.0009
  K=4096: acc=0.8061±0.0187, erank=15.40, S=0.0038, marginal=+0.0053

CIFAR_3v5 | 3 vs 5 | 50 trials
  K=   2: acc=0.5048±0.0325, erank=1.82, S=0.9096, marginal=+0.0000
  K=   4: acc=0.5068±0.0330, erank=4.22, S=

## Statistical Significance Tests

In [18]:
from scipy.stats import wilcoxon, spearmanr, pearsonr

# ============================================================
# TEST 1: Wilcoxon Signed-Rank Test for Oversampling Harm
# ============================================================

def get_raw_accuracies(X, y, K, test_size=200, n_trials=50):
    """Run sample_and_evaluate n_trials times and return raw accuracy array."""
    accs = []
    for trial in range(n_trials):
        acc, _ = sample_and_evaluate(X, y, K=K, test_size=test_size, seed=trial)
        accs.append(acc)
    return np.array(accs)

# --- MNIST 3v8 (the strongest effect) ---
mask = (y_mnist == 3) | (y_mnist == 8)
X_38 = X_mnist[mask]
y_38 = y_mnist[mask]
y_38 = np.where(y_38 == 3, 0, 1)

# Re-preprocess exactly as before
scaler = StandardScaler()
X_38_s = scaler.fit_transform(X_38)
pca = PCA(n_components=50)
X_38_pca = pca.fit_transform(X_38_s)

accs_2048 = get_raw_accuracies(X_38_pca, y_38, K=2048, test_size=200, n_trials=50)
accs_4096 = get_raw_accuracies(X_38_pca, y_38, K=4096, test_size=200, n_trials=50)

# One-tailed: is 2048 > 4096?
w_stat, p_val = wilcoxon(accs_2048, accs_4096, alternative='greater')
print(f"MNIST 3v8: W = {w_stat}, p = {p_val:.4f} (one-tailed, K=2048 > K=4096)")
print(f"  Mean acc: K=2048={accs_2048.mean():.4f}, K=4096={accs_4096.mean():.4f}")

# --- MNIST 0v1 (weak effect) ---
mask = (y_mnist == 0) | (y_mnist == 1)
X_01 = X_mnist[mask]
y_01 = y_mnist[mask]
y_01 = np.where(y_01 == 0, 0, 1)
scaler = StandardScaler()
X_01_s = scaler.fit_transform(X_01)
pca = PCA(n_components=50)
X_01_pca = pca.fit_transform(X_01_s)

accs_2048_01 = get_raw_accuracies(X_01_pca, y_01, K=2048, test_size=200, n_trials=50)
accs_4096_01 = get_raw_accuracies(X_01_pca, y_01, K=4096, test_size=200, n_trials=50)
w_stat_01, p_val_01 = wilcoxon(accs_2048_01, accs_4096_01, alternative='greater')
print(f"\nMNIST 0v1: W = {w_stat_01}, p = {p_val_01:.4f}")

# --- MNIST 4v9 (weak effect) ---
mask = (y_mnist == 4) | (y_mnist == 9)
X_49 = X_mnist[mask]
y_49 = y_mnist[mask]
y_49 = np.where(y_49 == 4, 0, 1)
scaler = StandardScaler()
X_49_s = scaler.fit_transform(X_49)
pca = PCA(n_components=50)
X_49_pca = pca.fit_transform(X_49_s)

accs_2048_49 = get_raw_accuracies(X_49_pca, y_49, K=2048, test_size=200, n_trials=50)
accs_4096_49 = get_raw_accuracies(X_49_pca, y_49, K=4096, test_size=200, n_trials=50)
w_stat_49, p_val_49 = wilcoxon(accs_2048_49, accs_4096_49, alternative='greater')
print(f"MNIST 4v9: W = {w_stat_49}, p = {p_val_49:.4f}")

MNIST 3v8: W = 702.0, p = 0.0718 (one-tailed, K=2048 > K=4096)
  Mean acc: K=2048=0.9659, K=4096=0.9630

MNIST 0v1: W = 474.0, p = 0.2850
MNIST 4v9: W = 491.0, p = 0.5186


In [19]:
# ============================================================
# TEST 2: Spearman Correlation — S(K) vs. Marginal Accuracy Gain
# ============================================================

all_S = []
all_marginal = []

for task_name, results in all_results.items():
    for i, r in enumerate(results):
        if i == 0:  # skip K=2, no previous K to compute marginal
            continue
        # Only include valid marginals (skip if previous was skipped)
        marginal = r['marginal']
        S = r['S']
        if marginal is not None and not np.isnan(marginal):
            all_S.append(S)
            all_marginal.append(marginal)

rho, p_value = spearmanr(all_S, all_marginal)

print(f"Spearman correlation across all tasks and K values:")
print(f"  rho = {rho:.3f}")
print(f"  p-value = {p_value:.2e}")
print(f"  N = {len(all_S)} observations")

# Also show by predicted phase
import pandas as pd
df = pd.DataFrame({'S': all_S, 'marginal': all_marginal})
df['phase'] = pd.cut(df['S'], bins=[0, 0.3, 1.0, np.inf], 
                     labels=['Saturation', 'Transition', 'Exploration'])
print("\nMean marginal gain by predicted phase:")
print(df.groupby('phase')['marginal'].agg(['mean', 'std', 'count']))

Spearman correlation across all tasks and K values:
  rho = 0.705
  p-value = 2.87e-13
  N = 80 observations

Mean marginal gain by predicted phase:
                 mean       std  count
phase                                 
Saturation   0.007016  0.009845     48
Transition   0.029454  0.022726     26
Exploration  0.050542  0.033404      6


In [20]:
# ============================================================
# TEST 3: Pearson Correlation — Decoupling (erank_∞ vs. Peak Accuracy)
# ============================================================

# Extract asymptote and peak accuracy for each task
# These are read from your results tables
eranks_inf = []
peaks = []
task_names = []

for name, results in all_results.items():
    # erank asymptote = last mean_erank
    erank_inf = results[-1]['mean_erank']
    # peak accuracy = max mean_acc
    peak_acc = max([r['mean_acc'] for r in results])
    
    eranks_inf.append(erank_inf)
    peaks.append(peak_acc)
    task_names.append(name)

r, p = pearsonr(eranks_inf, peaks)

print("Task-level asymptotes and peak accuracies:")
for n, e, p_acc in zip(task_names, eranks_inf, peaks):
    print(f"  {n:20s}: erank_inf={e:.2f}, peak_acc={p_acc:.4f}")

print(f"\nPearson correlation (erank_inf vs. peak_acc):")
print(f"  r = {r:.3f}")
print(f"  p = {p:.3f}")
print(f"  N = {len(eranks_inf)} tasks")
if p > 0.05:
    print("  Result: NOT SIGNIFICANT (supports decoupling hypothesis)")
else:
    print("  Result: SIGNIFICANT (would contradict decoupling)")

Task-level asymptotes and peak accuracies:
  MNIST_0v1           : erank_inf=32.24, peak_acc=0.9969
  MNIST_3v8           : erank_inf=37.45, peak_acc=0.9659
  MNIST_4v9           : erank_inf=35.58, peak_acc=0.9565
  Fashion_0v1         : erank_inf=20.69, peak_acc=0.9851
  Fashion_2v6         : erank_inf=14.46, peak_acc=0.8344
  Kuzushiji_0v1       : erank_inf=37.19, peak_acc=0.9873
  USPS_1v2            : erank_inf=15.08, peak_acc=0.9973
  BreastCancer        : erank_inf=8.20, peak_acc=0.9449

Pearson correlation (erank_inf vs. peak_acc):
  r = 0.391
  p = 0.338
  N = 8 tasks
  Result: NOT SIGNIFICANT (supports decoupling hypothesis)


## Ablation


In [21]:
for d in [20, 50, 100]:
    pca = PCA(n_components=d)
    X_pca_d = pca.fit_transform(X_38_s)  # X_38_s is the standardized MNIST 3v8 data
    results_d = run_experiment(f'MNIST_3v8_d{d}', X_pca_d, y_38, 0, 1,
        Ks=[2,4,8,16,32,64,128,256,512,1024,2048,4096], test_size=200, n_trials=20, pca_dims=None)


MNIST_3v8_d20 | 0 vs 1 | 20 trials
  K=   2: acc=0.6264±0.0654, erank=1.78, S=0.8916, marginal=+0.0000
  K=   4: acc=0.6940±0.0561, erank=4.38, S=1.0958, marginal=+0.0676
  K=   8: acc=0.7853±0.0468, erank=7.57, S=0.9462, marginal=+0.0912
  K=  16: acc=0.8557±0.0381, erank=11.99, S=0.7495, marginal=+0.0705
  K=  32: acc=0.8969±0.0212, erank=14.03, S=0.4385, marginal=+0.0411
  K=  64: acc=0.9052±0.0191, erank=16.28, S=0.2543, marginal=+0.0084
  K= 128: acc=0.9321±0.0157, erank=17.64, S=0.1378, marginal=+0.0269
  K= 256: acc=0.9471±0.0126, erank=18.38, S=0.0718, marginal=+0.0150
  K= 512: acc=0.9536±0.0119, erank=19.04, S=0.0372, marginal=+0.0065
  K=1024: acc=0.9563±0.0086, erank=19.32, S=0.0189, marginal=+0.0026
  K=2048: acc=0.9577±0.0083, erank=19.46, S=0.0095, marginal=+0.0015
  K=4096: acc=0.9525±0.0075, erank=19.59, S=0.0048, marginal=-0.0052

MNIST_3v8_d50 | 0 vs 1 | 20 trials
  K=   2: acc=0.5952±0.0602, erank=1.83, S=0.9140, marginal=+0.0000
  K=   4: acc=0.6406±0.0488, erank=

In [27]:
def sample_and_evaluate_with_C(X, y, K, test_size=200, seed=None, C=np.inf):
    """Wrapper that passes C through to LogisticRegression."""
    rng = np.random.default_rng(seed)
    
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]
    
    train_0 = rng.choice(idx_0, size=K, replace=False)
    train_1 = rng.choice(idx_1, size=K, replace=False)
    
    rem_0 = np.setdiff1d(idx_0, train_0)
    rem_1 = np.setdiff1d(idx_1, train_1)
    
    test_0 = rng.choice(rem_0, size=test_size, replace=False)
    test_1 = rng.choice(rem_1, size=test_size, replace=False)
    
    X_train = np.vstack([X[train_0], X[train_1]])
    y_train = np.array([0]*K + [1]*K)
    X_test = np.vstack([X[test_0], X[test_1]])
    y_test = np.array([0]*test_size + [1]*test_size)
    
    X_train_mean = X_train.mean(axis=0)
    X_train_centered = X_train - X_train_mean
    X_test_centered = X_test - X_train_mean
    
    clf = LogisticRegression(max_iter=5000, C=C)
    clf.fit(X_train_centered, y_train)
    acc = clf.score(X_test_centered, y_test)
    
    cov_0 = np.cov(X_train_centered[:K], rowvar=False, bias=True)
    cov_1 = np.cov(X_train_centered[K:], rowvar=False, bias=True)
    cov_pooled = 0.5 * (cov_0 + cov_1)
    
    eigvals = np.linalg.eigvalsh(cov_pooled)
    eigvals = np.maximum(eigvals, 1e-12)
    p = eigvals / eigvals.sum()
    erank = np.exp(-np.sum(p * np.log(p)))
    
    return acc, erank



In [28]:
for C in [np.inf, 1.0, 0.1]:
    accs = []
    for trial in range(50):
        acc, _ = sample_and_evaluate_with_C(X_38_pca, y_38, K=4096, test_size=200, seed=trial, C=C)
        accs.append(acc)
    C_str = "inf" if C == np.inf else str(C)
    print(f"C={C_str:>6s}: acc={np.mean(accs):.4f}±{np.std(accs):.4f}")

C=   inf: acc=0.9630±0.0096
C=   1.0: acc=0.9628±0.0095
C=   0.1: acc=0.9631±0.0093
